# Devkota: train on Colab

Runtime → Change runtime type → **T4 GPU**.

Before running, upload `train_merged.txt`, `validation_merged.txt` and `poem.txt` to `/content/` (Files panel).

Everything lives in `/content`, which is wiped if the runtime disconnects, so download checkpoints as you go.

## 1. Code + data
Moves the uploaded files into the layout `main.py` expects: `data/merged/` and `data/preprocessed/`.

In [ ]:
%cd /content
!test -d devkota || git clone https://github.com/un1u3/devkota.git
!cd devkota && git pull
!pip install -q sentencepiece

!mkdir -p /content/data/merged /content/data/preprocessed
!test -f /content/train_merged.txt && mv /content/train_merged.txt /content/validation_merged.txt /content/data/merged/ || true
!test -f /content/poem.txt && mv /content/poem.txt /content/data/preprocessed/ || true
!ls /content/data/merged /content/data/preprocessed

%cd /content/devkota/devkota-v0.7
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# check the uploads are complete (sizes and md5 of the originals)
import hashlib, os
expected = {
    '/content/data/merged/train_merged.txt': (60393441, '3cb53eabdca7727d63da1f2981f31368'),
    '/content/data/merged/validation_merged.txt': (26501849, 'bbcf08c9154e3ff2b0ec0b4a303b5a13'),
    '/content/data/preprocessed/poem.txt': (535837, '93240f950a4a1707ace310ed366a716b'),
}
for path, (size, md5) in expected.items():
    if not os.path.exists(path):
        print('MISSING ', path); continue
    got = os.path.getsize(path)
    ok = got == size and hashlib.md5(open(path, 'rb').read()).hexdigest() == md5
    print('ok      ' if ok else f'BAD     ({got:,} of {size:,} bytes) re-upload:', path)

## 2. Tokenize

In [ ]:
!python main.py prepare --data-dir /content/data

## 3. Pretrain on the general Nepali corpus
Checkpoints go to `devkota-v0.7/checkpoints/`.

In [ ]:
!python main.py pretrain --data-dir /content/data \
    --d-model 384 --num-layers 6 --num-heads 6 --max-seq-len 256 \
    --batch-size 32 --steps 8000 --eval-every 500 --lr 3e-4

Download the pretrained model now, so a disconnect during fine-tuning doesn't cost you the long run.

In [ ]:
from google.colab import files
files.download('checkpoints/pretrain.pt')

## 4. Fine-tune on the poems
Starts from `checkpoints/pretrain.pt`.

In [ ]:
!python main.py finetune --data-dir /content/data \
    --steps 1000 --eval-every 100 --lr 1e-4

## 5. Generate

In [ ]:
!python main.py generate --prompt "हिमाल" --max-tokens 200 --temperature 0.8

## 6. Download the poem model

In [ ]:
from google.colab import files
files.download('checkpoints/devkota_poet.pt')